In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
)

import numpy as np
import pandas as pd

import matplotlib
matplotlib.rcParams["svg.fonttype"] = "none"

import numpy as np
import pandas as pd
import pathlib as pl

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
)

# ============================================================
# Utilities
# ============================================================

def patient_scale(bags):
    """
    Per-patient z-score normalization.
    """

    scaled = {}

    for pid, X in bags.items():

        mean = X.mean(axis=0, keepdims=True)
        std = X.std(axis=0, keepdims=True)

        scaled[pid] = (X - mean) / (std + 1e-6)

    return scaled


def group_small_clusters(
    df,
    cluster_col,
    min_count=1000,
    new_label="Other",
):

    counts = df[cluster_col].value_counts()

    small = counts[counts < min_count].index

    grouped = df[cluster_col].astype(str).copy()

    grouped.loc[grouped.isin(small)] = new_label

    return grouped


# ============================================================
# Features
# ============================================================

def bootstrap_patient_feature(
    X,
    n_cells=1000,
    mean_std=True,
    rng=None,
):

    idx = rng.choice(
        len(X),
        size=n_cells,
        replace=(len(X) < n_cells),
    )

    X_sub = X[idx]

    if mean_std:

        return np.concatenate(
            [
                X_sub.mean(axis=0),
                X_sub.std(axis=0),
            ]
        )

    return X_sub.mean(axis=0)


def full_patient_feature(
    X,
    mean_std=True,
):

    if mean_std:

        return np.concatenate(
            [
                X.mean(axis=0),
                X.std(axis=0),
            ]
        )

    return X.mean(axis=0)


# ============================================================
# LOO bootstrap LR + RF
# ============================================================

def loo_bootstrap_lr(
    bags,
    labels_dict,
    mode="bootstrap",
    n_bootstrap=150,
    n_cells=1000,
    mean_std=True,
    C=0.01,
    random_state=0,
):

    rng = np.random.default_rng(random_state)

    patient_ids = sorted(bags.keys())

    results = []

    for left_out in patient_ids:

        train_ids = [
            pid
            for pid in patient_ids
            if pid != left_out
        ]

        X_train = []
        y_train = []
        
        for pid in train_ids:
        
            if mode == "mean":
        
                X_train.append(
                    full_patient_feature(
                        bags[pid],
                        mean_std=mean_std,
                    )
                )
        
                y_train.append(
                    labels_dict[pid]
                )
        
            else:
        
                for _ in range(n_bootstrap):
        
                    X_train.append(
                        bootstrap_patient_feature(
                            bags[pid],
                            n_cells=n_cells,
                            mean_std=mean_std,
                            rng=rng,
                        )
                    )
        
                    y_train.append(
                        labels_dict[pid]
                    )

        X_train = np.vstack(X_train)
        y_train = np.array(y_train)

        clf = LogisticRegression(
            class_weight="balanced",
            penalty="l2",
            C=C,
            max_iter=10000,
        )

        clf.fit(
            X_train,
            y_train,
        )

        X_test = (
            full_patient_feature(
                bags[left_out],
                mean_std=mean_std,
            )
            .reshape(1, -1)
        )

        prob = clf.predict_proba(
            X_test
        )[0, 1]

        pred = int(prob > 0.5)

        results.append(
            {
                "sample": left_out,
                "true": labels_dict[left_out],
                "prob": prob,
                "pred": pred,
            }
        )

    return pd.DataFrame(results)

from sklearn.ensemble import RandomForestClassifier

def loo_bootstrap_rf(
    bags,
    labels_dict,
    mode='bootstrap',
    n_bootstrap=150,
    n_cells=1000,
    mean_std=False,
    n_estimators=100,
    max_depth=3,
    random_state=0,
):

    rng = np.random.default_rng(random_state)

    patient_ids = sorted(bags.keys())

    results = []

    for left_out in patient_ids:

        train_ids = [
            pid
            for pid in patient_ids
            if pid != left_out
        ]

        X_train = []
        y_train = []

        for pid in train_ids:
        
            if mode == "mean":
        
                X_train.append(
                    full_patient_feature(
                        bags[pid],
                        mean_std=mean_std,
                    )
                )
        
                y_train.append(
                    labels_dict[pid]
                )
        
            else:
        
                for _ in range(n_bootstrap):
        
                    X_train.append(
                        bootstrap_patient_feature(
                            bags[pid],
                            n_cells=n_cells,
                            mean_std=mean_std,
                            rng=rng,
                        )
                    )
        
                    y_train.append(
                        labels_dict[pid]
                    )

        X_train = np.vstack(X_train)
        y_train = np.array(y_train)

        # ----------------------------------
        # Random Forest
        # ----------------------------------

        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )

        clf.fit(
            X_train,
            y_train,
        )

        # ----------------------------------
        # Full patient test representation
        # ----------------------------------

        X_test = (
            full_patient_feature(
                bags[left_out],
                mean_std=mean_std,
            )
            .reshape(1, -1)
        )

        prob = clf.predict_proba(
            X_test
        )[0, 1]

        pred = int(prob > 0.5)

        results.append(
            {
                "sample": left_out,
                "true": labels_dict[left_out],
                "prob": prob,
                "pred": pred,
            }
        )

    return pd.DataFrame(results)

# ============================================================
# Load metadata
# ============================================================

clinical_info = pd.read_csv(
    "../../../Broad_SpatialFoundation/VisiumHD-LUAD/clinical-info/full_clinical.csv",
    index_col=0,
)

adata_obs = pd.read_parquet(
    "../../../Broad_SpatialFoundation/notebooks/nsclc_adata_obs.parquet"
)

adata_obs["leiden_joint"] = group_small_clusters(
    adata_obs,
    cluster_col="leiden",
    min_count=1000,
)

# ============================================================
# Load NicheFinder embeddings
# ============================================================

base_dir = pl.Path(
    "../../../Broad_SpatialFoundation/VisiumHD-LUAD-processed/"
)

sample_list = np.setdiff1d(
    [f.stem for f in base_dir.iterdir()],
    ["full_cohort", "LIB-064888st1"],
)

malignant_niches = [
    "0",
    "1",
    "2",
    "4",
    "5",
    "7",
    "10",
    "12",
    "15",
    "16",
]

embeddings = {}

for sample in sample_list:

    emb = pd.read_parquet(
        base_dir / sample / "embeddings" / "NicheFinder.parquet"
    )

    emb.columns = emb.columns.astype(str)

    emb = emb[[str(i) for i in range(10)]]

    emb.index = emb.index + "::" + sample

    emb = emb.loc[
        emb.index.intersection(adata_obs.index)
    ]

    embeddings[sample] = emb

# ============================================================
# Select malignant niches
# ============================================================

sub_embeddings = {}

for sample in embeddings:

    sample_obs = adata_obs.loc[
        adata_obs["sample_id"] == sample,
        ["leiden_joint"],
    ]

    common = embeddings[sample].index.intersection(
        sample_obs.index
    )

    emb = embeddings[sample].loc[common]

    sample_obs = sample_obs.loc[common]

    mask = sample_obs["leiden_joint"].isin(
        malignant_niches
    )

    sub_embeddings[sample] = emb.loc[mask]

# ============================================================
# Labels
# ============================================================

target_column = "pTNM T red"

targets = (
    clinical_info[
        ["Library", target_column]
    ]
    .set_index("Library")
)

targets = pd.get_dummies(
    targets
).astype(int)

targets = targets.iloc[:, 0]

labels_dict = targets.to_dict()

# ============================================================
# Bags
# ============================================================

bags = {}

for sample, emb in sub_embeddings.items():

    if sample not in labels_dict:
        continue

    if len(emb) == 0:
        continue

    bags[sample] = emb.to_numpy(
        dtype=np.float32
    )

print(
    f"Retained {len(bags)} patients"
)

for k, v in bags.items():
    print(k, v.shape)

# LR

In [ ]:
# ============================================================
# Hyperparameter sweep
# ============================================================

summary_results = []
all_predictions = []

for mode in ["mean", "bootstrap"]:

    for mean_std in [False, True]:
    
        for C in [0.1, 1, 10, 100]:
    
            for n_bootstrap in [50, 100, 200]:

                if mode=="mean" and n_bootstrap>50:
                    continue
    
                print(
                    f"\nmean_std={mean_std} "
                    f"C={C} "
                    f"n_bootstrap={n_bootstrap} "
                    f"mode={mode}"
                )
    
                pred_df = loo_bootstrap_lr(
                    bags=bags,
                    labels_dict=labels_dict,
                    mode=mode,
                    n_bootstrap=n_bootstrap,
                    n_cells=1000,
                    mean_std=mean_std,
                    C=C,
                    random_state=0,
                )
    
                auc = roc_auc_score(
                    pred_df.true,
                    pred_df.prob,
                )
    
                bac = balanced_accuracy_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                f1 = f1_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                summary_results.append(
                    {
                        "mode": mode,
                        "mean_std": mean_std,
                        "C": C,
                        "n_bootstrap": n_bootstrap,
                        "AUC": auc,
                        "BAC": bac,
                        "F1": f1,
                    }
                )
    
                tmp = pred_df.copy()
    
                tmp["mean_std"] = mean_std
                tmp["C"] = C
                tmp["n_bootstrap"] = n_bootstrap
                tmp["mode"] = mode
    
                all_predictions.append(tmp)
    
                print(
                    f"AUC={auc:.3f} "
                    f"BAC={bac:.3f} "
                    f"F1={f1:.3f}"
                )

# ============================================================
# Results tables
# ============================================================

summary_df = pd.DataFrame(summary_results)

all_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True,
)

summary_df = summary_df.sort_values(
    "AUC",
    ascending=False,
)

print("\n")
print("=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    summary_df.round(3)
)

# ============================================================
# Save
# ============================================================

summary_df.to_csv(
    "lr_hyperparameter_sweep_summary.csv",
    index=False,
)

all_predictions_df.to_csv(
    "lr_hyperparameter_sweep_predictions.csv",
    index=False,
)

In [ ]:
plot_df = summary_df.copy()

plot_df["representation"] = np.where(
    plot_df["mode"] == "mean",
    "Mean",
    "Boot" + plot_df["n_bootstrap"].astype(str)
)

plot_df["representation"] = pd.Categorical(
    plot_df["representation"],
    categories=["Mean", "Boot50", "Boot100", "Boot200"],
    ordered=True,
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="C",
            columns="representation",
            values="AUC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("C")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "lr_auc_heatmap.svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="C",
            columns="representation",
            values="BAC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("C")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "lr_bac_heatmap.svg",
    bbox_inches="tight",
)
plt.show()

# RF

In [ ]:
# ============================================================
# Hyperparameter sweep
# ============================================================

summary_results = []
all_predictions = []

for mode in ["mean", "bootstrap"]:

    for mean_std in [False, True]:
    
        for n_estimators in [100, 200, 300]:
    
            for n_bootstrap in [50, 100, 200]:

                if mode=="mean" and n_bootstrap>50:
                    continue
    
                print(
                    f"\nmean_std={mean_std} "
                    f"n_estimators={n_estimators} "
                    f"n_bootstrap={n_bootstrap} "
                    f"mode={mode}"
                )
    
                pred_df = loo_bootstrap_rf(
                    bags=bags,
                    labels_dict=labels_dict,
                    mode=mode,
                    n_bootstrap=n_bootstrap,
                    n_cells=1000,
                    mean_std=mean_std,
                    n_estimators=n_estimators,
                    random_state=0,
                )
    
                auc = roc_auc_score(
                    pred_df.true,
                    pred_df.prob,
                )
    
                bac = balanced_accuracy_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                f1 = f1_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                summary_results.append(
                    {
                        "mode": mode,
                        "mean_std": mean_std,
                        "n_estimators": n_estimators,
                        "n_bootstrap": n_bootstrap,
                        "AUC": auc,
                        "BAC": bac,
                        "F1": f1,
                    }
                )
    
                tmp = pred_df.copy()
    
                tmp["mean_std"] = mean_std
                tmp["n_estimators"] = n_estimators
                tmp["n_bootstrap"] = n_bootstrap
                tmp["mode"] = mode
    
                all_predictions.append(tmp)
    
                print(
                    f"AUC={auc:.3f} "
                    f"BAC={bac:.3f} "
                    f"F1={f1:.3f}"
                )

# ============================================================
# Results tables
# ============================================================

summary_df = pd.DataFrame(summary_results)

all_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True,
)

summary_df = summary_df.sort_values(
    "AUC",
    ascending=False,
)

print("\n")
print("=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    summary_df.round(3)
)

# ============================================================
# Save
# ============================================================

summary_df.to_csv(
    "rf_hyperparameter_sweep_summary.csv",
    index=False,
)

all_predictions_df.to_csv(
    "rf_hyperparameter_sweep_predictions.csv",
    index=False,
)

In [ ]:
plot_df = summary_df.copy()

plot_df["representation"] = np.where(
    plot_df["mode"] == "mean",
    "Mean",
    "Boot" + plot_df["n_bootstrap"].astype(str)
)

plot_df["representation"] = pd.Categorical(
    plot_df["representation"],
    categories=["Mean", "Boot50", "Boot100", "Boot200"],
    ordered=True,
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="n_estimators",
            columns="representation",
            values="AUC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("N estimators")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "rf_auc_heatmap.svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="n_estimators",
            columns="representation",
            values="BAC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("N estimators")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "rf_bac_heatmap.svg",
    bbox_inches="tight",
)
plt.show()

# Comparing with GEX

In [ ]:
from tqdm.auto import tqdm
import scanpy as sc

In [ ]:
base_dir = pl.Path('../../../Broad_SpatialFoundation/VisiumHD-LUAD-processed/')
sample_list = np.setdiff1d([f.stem for f in base_dir.iterdir()],['full_cohort','LIB-064888st1'])
sample_list

In [ ]:
adatas = {}
for sample in tqdm(sample_list):
    adata = sc.read_h5ad(base_dir / sample / 'adata.h5ad')
    adata.obs_names = adata.obs_names + '::' + sample 
    adata.obs['sample_id'] = sample
    
    common_idx = adata.obs_names.intersection(embeddings_df[sample].index)
    adata = adata[common_idx].copy()
    adata = adata[adata.obs.celltypes != 'Noise'].copy()
    adatas[sample] = adata

In [ ]:
gex_embeddings = {}
for sample in tqdm(sample_list):
    sc.pp.normalize_total(adatas[sample], target_sum=10000)
    sc.pp.log1p(adatas[sample])
    sc.tl.pca(adatas[sample], )
    gex_embeddings[sample] = adatas[sample].obsm['X_pca'].copy()
    gex_embeddings[sample] = pd.DataFrame(gex_embeddings[sample], index=adatas[sample].obs_names)

In [ ]:
# ============================================================
# Load metadata
# ============================================================

clinical_info = pd.read_csv(
    "../../../Broad_SpatialFoundation/VisiumHD-LUAD/clinical-info/full_clinical.csv",
    index_col=0,
)

adata_obs = pd.read_parquet(
    "../../../Broad_SpatialFoundation/notebooks/nsclc_adata_obs.parquet"
)

adata_obs["leiden_joint"] = group_small_clusters(
    adata_obs,
    cluster_col="leiden",
    min_count=1000,
)

# ============================================================
# Load NicheFinder embeddings
# ============================================================

base_dir = pl.Path(
    "../../../Broad_SpatialFoundation/VisiumHD-LUAD-processed/"
)

sample_list = np.setdiff1d(
    [f.stem for f in base_dir.iterdir()],
    ["full_cohort", "LIB-064888st1"],
)

malignant_niches = [
    "0",
    "1",
    "2",
    "4",
    "5",
    "7",
    "10",
    "12",
    "15",
    "16",
]

# ============================================================
# Select malignant niches
# ============================================================

sub_embeddings = {}

for sample in gex_embeddings:

    sample_obs = adata_obs.loc[
        adata_obs["sample_id"] == sample,
        ["leiden_joint"],
    ]

    common = gex_embeddings[sample].index.intersection(
        sample_obs.index
    )

    emb = gex_embeddings[sample].loc[common]

    sample_obs = sample_obs.loc[common]

    mask = sample_obs["leiden_joint"].isin(
        malignant_niches
    )

    sub_embeddings[sample] = emb.loc[mask]

# ============================================================
# Labels
# ============================================================

target_column = "pTNM T red"

targets = (
    clinical_info[
        ["Library", target_column]
    ]
    .set_index("Library")
)

targets = pd.get_dummies(
    targets
).astype(int)

targets = targets.iloc[:, 0]

labels_dict = targets.to_dict()

# ============================================================
# Bags
# ============================================================

bags = {}

for sample, emb in sub_embeddings.items():

    if sample not in labels_dict:
        continue

    if len(emb) == 0:
        continue

    bags[sample] = emb.to_numpy(
        dtype=np.float32
    )

print(
    f"Retained {len(bags)} patients"
)

for k, v in bags.items():
    print(k, v.shape)

In [ ]:
# ============================================================
# Hyperparameter sweep
# ============================================================

summary_results = []
all_predictions = []

for mode in ["mean", "bootstrap"]:

    for mean_std in [False, True]:
    
        for C in [0.1, 1, 10, 100]:
    
            for n_bootstrap in [50, 100, 200]:

                if mode=="mean" and n_bootstrap>50:
                    continue
    
                print(
                    f"\nmean_std={mean_std} "
                    f"C={C} "
                    f"n_bootstrap={n_bootstrap} "
                    f"mode={mode}"
                )
    
                pred_df = loo_bootstrap_lr(
                    bags=bags,
                    labels_dict=labels_dict,
                    mode=mode,
                    n_bootstrap=n_bootstrap,
                    n_cells=1000,
                    mean_std=mean_std,
                    C=C,
                    random_state=0,
                )
    
                auc = roc_auc_score(
                    pred_df.true,
                    pred_df.prob,
                )
    
                bac = balanced_accuracy_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                f1 = f1_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                summary_results.append(
                    {
                        "mode": mode,
                        "mean_std": mean_std,
                        "C": C,
                        "n_bootstrap": n_bootstrap,
                        "AUC": auc,
                        "BAC": bac,
                        "F1": f1,
                    }
                )
    
                tmp = pred_df.copy()
    
                tmp["mean_std"] = mean_std
                tmp["C"] = C
                tmp["n_bootstrap"] = n_bootstrap
                tmp["mode"] = mode
    
                all_predictions.append(tmp)
    
                print(
                    f"AUC={auc:.3f} "
                    f"BAC={bac:.3f} "
                    f"F1={f1:.3f}"
                )

# ============================================================
# Results tables
# ============================================================

summary_df = pd.DataFrame(summary_results)

all_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True,
)

summary_df = summary_df.sort_values(
    "AUC",
    ascending=False,
)

print("\n")
print("=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    summary_df.round(3)
)

# ============================================================
# Save
# ============================================================

summary_df.to_csv(
    "GEX_lr_hyperparameter_sweep_summary.csv",
    index=False,
)

all_predictions_df.to_csv(
    "GEX_lr_hyperparameter_sweep_predictions.csv",
    index=False,
)

In [ ]:
plot_df = summary_df.copy()

plot_df["representation"] = np.where(
    plot_df["mode"] == "mean",
    "Mean",
    "Boot" + plot_df["n_bootstrap"].astype(str)
)

plot_df["representation"] = pd.Categorical(
    plot_df["representation"],
    categories=["Mean", "Boot50", "Boot100", "Boot200"],
    ordered=True,
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="C",
            columns="representation",
            values="AUC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("C")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "GEX_lr_auc_heatmap.svg",
    bbox_inches="tight",
)
plt.show()

## RF

In [ ]:
# ============================================================
# Hyperparameter sweep
# ============================================================

summary_results = []
all_predictions = []

for mode in ["mean", "bootstrap"]:

    for mean_std in [False, True]:
    
        for n_estimators in [100, 200, 300]:
    
            for n_bootstrap in [50, 100, 200]:

                if mode=="mean" and n_bootstrap>50:
                    continue
    
                print(
                    f"\nmean_std={mean_std} "
                    f"n_estimators={n_estimators} "
                    f"n_bootstrap={n_bootstrap} "
                    f"mode={mode}"
                )
    
                pred_df = loo_bootstrap_rf(
                    bags=bags,
                    labels_dict=labels_dict,
                    mode=mode,
                    n_bootstrap=n_bootstrap,
                    n_cells=1000,
                    mean_std=mean_std,
                    n_estimators=n_estimators,
                    random_state=0,
                )
    
                auc = roc_auc_score(
                    pred_df.true,
                    pred_df.prob,
                )
    
                bac = balanced_accuracy_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                f1 = f1_score(
                    pred_df.true,
                    pred_df.pred,
                )
    
                summary_results.append(
                    {
                        "mode": mode,
                        "mean_std": mean_std,
                        "n_estimators": n_estimators,
                        "n_bootstrap": n_bootstrap,
                        "AUC": auc,
                        "BAC": bac,
                        "F1": f1,
                    }
                )
    
                tmp = pred_df.copy()
    
                tmp["mean_std"] = mean_std
                tmp["n_estimators"] = n_estimators
                tmp["n_bootstrap"] = n_bootstrap
                tmp["mode"] = mode
    
                all_predictions.append(tmp)
    
                print(
                    f"AUC={auc:.3f} "
                    f"BAC={bac:.3f} "
                    f"F1={f1:.3f}"
                )

# ============================================================
# Results tables
# ============================================================

summary_df = pd.DataFrame(summary_results)

all_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True,
)

summary_df = summary_df.sort_values(
    "AUC",
    ascending=False,
)

print("\n")
print("=" * 80)
print("SUMMARY")
print("=" * 80)

print(
    summary_df.round(3)
)

# ============================================================
# Save
# ============================================================

summary_df.to_csv(
    "GEX_rf_hyperparameter_sweep_summary.csv",
    index=False,
)

all_predictions_df.to_csv(
    "GEX_rf_hyperparameter_sweep_predictions.csv",
    index=False,
)

In [ ]:
plot_df = summary_df.copy()

plot_df["representation"] = np.where(
    plot_df["mode"] == "mean",
    "Mean",
    "Boot" + plot_df["n_bootstrap"].astype(str)
)

plot_df["representation"] = pd.Categorical(
    plot_df["representation"],
    categories=["Mean", "Boot50", "Boot100", "Boot200"],
    ordered=True,
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    constrained_layout=True,
)

for ax, ms in zip(axes, [False, True]):

    mat = (
        plot_df
        .query("mean_std == @ms")
        .pivot(
            index="n_estimators",
            columns="representation",
            values="AUC",
        )
        .sort_index()
    )

    sns.heatmap(
        mat,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=0.5,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )

    # separator between Mean and Bootstrap
    ax.axvline(1, color="white", lw=4)

    ax.set_title(
        "Mean only features"
        if not ms
        else "Mean + SD features"
    )

    ax.set_ylabel("N estimators")
    ax.set_xlabel("Patient representation")

plt.savefig(
    "GEX_rf_auc_heatmap.svg",
    bbox_inches="tight",
)
plt.show()